# Week 8: RAG + Summarisation

SiteLens AI — retrieve precedents, summarise findings in inspector-speak.

**Topic:** NLP, RAG, abstractive summarisation  
**Dates:** May 10–14 2026  
**Deliverable:** End-to-end pipeline: query → retrieve top-k → summarise with t5-small.

---

| Cell | Type | Purpose |
|---|---|---|
| header | md | This cell |
| 1 | code | SETUP — imports, env, clients |
| 2 | code | DATA — load sample records from JSON |
| 3 | code | EMBED — load sentence-transformer model |
| 4 | code | RETRIEVE — query top-k nearest neighbours |
| 5 | code | SUMMARISE — abstractive summary with t5-small |
| 6 | code | INSPECT — run multiple queries, print full results |

In [ ]:
# 1 SETUP — imports, env, clients
import sys, os, json
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone
from transformers import pipeline

load_dotenv('../.env')
PINECONE_API_KEY = os.getenv('PINECONE_API_KEY')
INDEX_NAME = os.getenv('PINECONE_INDEX_NAME', 'sitelens')

assert PINECONE_API_KEY, 'PINECONE_API_KEY not loaded — check .env path'
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(INDEX_NAME)
print('Connected to index:', INDEX_NAME)
print(index.describe_index_stats())

In [ ]:
# 2 DATA — load sample records from committed JSON
with open('../data/samples/sample_records.json', encoding='utf-8') as f:
    records = json.load(f)

print(f'Loaded {len(records)} records')
for r in records[:2]:
    print(r['text'])

In [ ]:
# 3 EMBED — load sentence-transformer model
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print('Embedding model ready.')

In [ ]:
# 4 RETRIEVE — query top-k nearest neighbours
def retrieve(query: str, top_k: int = 3):
    vec = embed_model.encode([query])[0].tolist()
    results = index.query(vector=vec, top_k=top_k, include_metadata=True)
    return results['matches']

query = 'building destroyed by fire in dense urban area'
matches = retrieve(query)
print(f"Query: '{query}'")
for m in matches:
    print(f"  [{m['score']:.3f}] {m['id']}  {m['metadata']['text']}")

In [ ]:
# 5 SUMMARISE — abstractive summary with t5-small (direct, no pipeline)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained('t5-small')
t5 = AutoModelForSeq2SeqLM.from_pretrained('t5-small')
print('Summariser ready.')

combined = 'summarize: ' + ' '.join([m['metadata']['text'] for m in matches])
inputs = tokenizer(combined, return_tensors='pt', max_length=512, truncation=True)
outputs = t5.generate(inputs.input_ids, max_length=80, min_length=20, num_beams=4)
summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
print('\nSummary:')
print(summary)

In [ ]:
# 6 INSPECT — global summary + per-scenario narratives
#
# Dataset note: GSI hazard zones are mutually exclusive in this dataset
# (fire=311, tsunami=3418, slope=169 buildings — no building carries more than one flag).
# Mixed-hazard summaries arise when an inspector covers multiple disconnected areas.
# Scenario 4 simulates that case.

import re
from collections import Counter

def parse_hits(hits):
    outcomes, hazards, mmis, evidences = [], set(), [], set()
    for h in hits:
        for part in h['metadata']['text'].split('. '):
            if part.startswith('Building '):
                outcomes.append(part.replace('Building ', '').lower())
            elif part.startswith('Hazard: '):
                hazards.add(part.replace('Hazard: ', ''))
            elif 'MMI' in part:
                m = re.search(r'MMI ([\d.]+)', part)
                if m:
                    mmis.append(float(m.group(1)))
            elif part.startswith('Evidence: '):
                evidences.add(part.replace('Evidence: ', '').replace(' assessment', '').rstrip('.'))
    return outcomes, hazards, mmis, evidences

def narrative_sentence(hits):
    outcomes, hazards, mmis, evidences = parse_hits(hits)
    n         = len(hits)
    counts    = Counter(outcomes)
    dominant  = counts.most_common(1)[0][0] if counts else 'assessed'
    haz_list  = sorted(h for h in hazards if h != 'seismic only')
    mmi_avg   = sum(mmis) / len(mmis) if mmis else 0
    mmi_label = 'severe' if mmi_avg >= 8 else 'strong' if mmi_avg >= 6 else 'moderate'
    ev_str    = ' and '.join(sorted(evidences))
    multi_zone = len(haz_list) > 1

    if haz_list:
        zone_str = ' and '.join(haz_list)
        cond_str = (f"across {zone_str} zones under {mmi_label} shaking"
                    if multi_zone else
                    f"under {mmi_label} {zone_str} conditions")
    else:
        cond_str = f"under {mmi_label} seismic shaking"

    destroyed = counts.get('destroyed', 0)
    survived  = counts.get('survived', 0)

    if dominant == 'destroyed':
        subject = f"All {n} buildings" if destroyed == n else f"{destroyed} of {n} buildings"
        s = f"{subject} were destroyed {cond_str} (MMI {mmi_avg:.1f})"
        if survived:
            s += f"; {survived} survived"
    elif dominant == 'survived':
        subject = f"All {n} buildings" if survived == n else f"{survived} of {n} buildings"
        s = f"{subject} survived {cond_str} (MMI {mmi_avg:.1f})"
        if destroyed:
            s += f"; {destroyed} were destroyed"
    else:
        s = f"{n} buildings recorded {cond_str} (MMI {mmi_avg:.1f})"

    return f"{s}. Evidence: {ev_str}."

def stats_line(hits):
    outcomes, hazards, mmis, evidences = parse_hits(hits)
    dominant = Counter(outcomes).most_common(1)[0][0] if outcomes else 'assessed'
    haz_list = sorted(h for h in hazards if h != 'seismic only')
    haz_str  = ' and '.join(haz_list) if haz_list else 'seismic'
    mmi_str  = f"MMI {sum(mmis)/len(mmis):.1f}" if mmis else ''
    ev_str   = ' and '.join(sorted(evidences))
    return f"{len(hits)} building(s) {dominant}: {haz_str} hazard, {mmi_str}, {ev_str} evidence."

scenarios = [
    {"label": "Scenario 1 - fire zone destruction",
     "query": "building destroyed by fire in dense urban area", "top_k": 3},
    {"label": "Scenario 2 - tsunami zone survival",
     "query": "structure survived tsunami zone with strong shaking", "top_k": 3},
    {"label": "Scenario 3 - obstructed assessment slope failure zone",
     "query": "obstructed assessment slope failure zone", "top_k": 3},
    {"label": "Scenario 4 - multi-area inspector (fire zone + tsunami zone)",
     "queries": [("building destroyed by fire dense urban area", 2),
                 ("building destroyed tsunami coastal zone", 2)]},
]

all_hits_pool, scenario_results = [], []
for scenario in scenarios:
    if "queries" in scenario:
        hits = []
        for q, k in scenario["queries"]:
            hits += retrieve(q, top_k=k)
    else:
        hits = retrieve(scenario["query"], top_k=scenario["top_k"])
    scenario_results.append((scenario["label"], hits))
    all_hits_pool.extend(hits)

global_outcomes, global_hazards, global_mmis, global_ev = parse_hits(all_hits_pool)
g_counts   = Counter(global_outcomes)
g_haz_list = sorted(h for h in global_hazards if h != 'seismic only')
g_haz      = ' and '.join(g_haz_list) if g_haz_list else 'seismic'
g_mmi_rng  = f"MMI {min(global_mmis):.1f}-{max(global_mmis):.1f}" if global_mmis else ''
g_ev       = ' and '.join(sorted(global_ev))
g_outcomes = ', '.join(f"{v} {k}" for k, v in g_counts.most_common())
print("=" * 60)
print("FIELD ASSESSMENT OVERVIEW")
print(f"  {len(all_hits_pool)} building records across {len(scenarios)} scenarios.")
print(f"  Hazard zones: {g_haz}. Shaking range: {g_mmi_rng}.")
print(f"  Outcomes: {g_outcomes}. Evidence: {g_ev}.")
print("=" * 60)

for label, hits in scenario_results:
    print("")
    print(label)
    print(" ", narrative_sentence(hits))
    print(" >>", stats_line(hits))
    for h in hits:
        print(f"  [{h['score']:.3f}] {h['id']}  {h['metadata']['text']}")